# AvgGRM-diversity: greedy versus GA+SA

This notebook compares two ways of choosing a size-$k$ training subset that maximizes the AvgGRM-diversity objective: the deterministic **greedy** rule and a **GA+SA metaheuristic**.

For each repeat, the notebook samples a fixed number of source candidates, runs both methods for several values of $k$ and $\lambda_{\mathrm{div}}$, and reports objective values, overlap, Jaccard similarity, and timing. Repeats therefore represent different subsampled candidate pools. Within one fixed candidate pool and fixed $\lambda_{\mathrm{div}}$, the greedy solution is deterministic.


In [ ]:
from pathlib import Path
import importlib.util
import json
import sys
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyreadr

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "scripts"))

PROJECT_ROOT


## Configuration

`N_CANDIDATES`, `K_VALUES`, `LAMBDA_DIVS`, and `N_REPEATS` set the comparison grid. Each repeat draws a new candidate pool from the source islands. The greedy solution is used as a warm start in the GA+SA population, so GA+SA should not finish below greedy apart from numerical roundoff.


In [ ]:
NPZ_PATH = PROJECT_ROOT / "Data" / "npz" / "snp_body_mass_ALL.npz"
GRM_RDS_PATH = PROJECT_ROOT / "Data" / "GRM" / "GRM_vanraden.rds"

MIN_COUNT = 20

# Use either a consecutive island code, e.g. 9, or set TARGET_ISLAND_LABEL to an original locality label such as "35".
TARGET_ISLAND_CODE = 9
TARGET_ISLAND_LABEL = None

# Larger experiment: n = 1800, k = 600, five lambda values, five repeats per
# lambda. Per-setting cost at this size is roughly 3.5x what it was at
# (1000, 300), so this grid keeps the total wall time close to one hour.
N_CANDIDATES = 1800
K_VALUES = [600]
LAMBDA_DIVS = [0.1, 0.25, 0.5, 1.0, 1.5]
N_REPEATS = 5

INCLUDE_DIAGONAL = True
SEED = 20260520
LOG_PROGRESS = True

RESULTS_DIR = PROJECT_ROOT / "outputs" / "final_results" / "avggrm_diversity_ga_sa"
FIGURES_DIR = PROJECT_ROOT / "figures"
THESIS_FIGURES_DIR = PROJECT_ROOT.parent / "69c525a2d91967a989023aaf" / "Figures"
PLOT_STEM = "avggrm_diversity_greedy_vs_ga_sa"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


## Load only what the comparison needs

The comparison uses IDs, locality, and the Van Raden GRM. It does not load the full SNP matrix.

In [ ]:
def load_ids_locality_grm(npz_path: Path, grm_rds_path: Path, min_count: int = 20):
    with np.load(npz_path, allow_pickle=False) as data:
        ids = np.asarray(data["ids"]).astype(str)
        locality = np.asarray(data["locality"]).astype(str).ravel()

    if ids.shape[0] != locality.shape[0]:
        raise ValueError("ids and locality have different lengths")

    grm_res = pyreadr.read_r(str(grm_rds_path))
    grm_df = next(iter(grm_res.values()))
    grm_df.index = grm_df.index.astype(str)
    grm_df.columns = grm_df.columns.astype(str)

    ids_in = np.array([sample_id for sample_id in ids if sample_id in grm_df.index], dtype=str)
    if ids_in.size == 0:
        raise ValueError("No overlapping IDs between NPZ and GRM")

    if ids_in.size != ids.size or not np.array_equal(ids_in, ids):
        pos = {sample_id: i for i, sample_id in enumerate(ids)}
        keep = np.array([pos[sample_id] for sample_id in ids_in], dtype=int)
        ids = ids[keep]
        locality = locality[keep]

    grm_df = grm_df.loc[ids, ids]

    # Match src.data.load_data: merge Ranes (68) into Lauvoya (67), then drop islands below min_count.
    locality = np.where(locality == "68", "67", locality)
    labels, counts = np.unique(locality, return_counts=True)
    keep_labels = set(labels[counts >= min_count])
    keep = np.array([label in keep_labels for label in locality], dtype=bool)

    ids = ids[keep]
    locality = locality[keep]
    grm_df = grm_df.loc[ids, ids]

    labels = np.array(sorted(np.unique(locality), key=lambda x: int(x)), dtype=str)
    label_to_code = {label: code for code, label in enumerate(labels)}
    code_to_label = {code: label for label, code in label_to_code.items()}
    locality_codes = np.array([label_to_code[label] for label in locality], dtype=int)

    return grm_df.to_numpy(dtype=float), ids, locality_codes, code_to_label


grm, ids, locality, code_to_label = load_ids_locality_grm(NPZ_PATH, GRM_RDS_PATH, MIN_COUNT)
counts = pd.Series(locality).value_counts().sort_index().rename_axis("island_code").reset_index(name="n")
counts["island_label"] = counts["island_code"].map(code_to_label)
display(counts)
print(f"Loaded GRM with shape {grm.shape}")

In [ ]:
if TARGET_ISLAND_LABEL is not None:
    label_to_code = {label: code for code, label in code_to_label.items()}
    target_code = int(label_to_code[str(TARGET_ISLAND_LABEL)])
else:
    target_code = int(TARGET_ISLAND_CODE)

target_idx = np.flatnonzero(locality == target_code)
source_idx_all = np.flatnonzero(locality != target_code)

if target_idx.size == 0:
    raise ValueError(f"No samples found for target island code {target_code}")
if source_idx_all.size < N_CANDIDATES:
    raise ValueError(f"Only {source_idx_all.size} source candidates available, fewer than N_CANDIDATES={N_CANDIDATES}")

print(f"Target island code: {target_code}, label: {code_to_label[target_code]}")
print(f"Target samples: {target_idx.size}")
print(f"Source candidates before subsampling: {source_idx_all.size}")

## Objective and greedy selection

In [ ]:
def q_value(selected, avg_grm_to_target, train_train_grm, lambda_div, include_diagonal=True):
    selected = np.asarray(selected, dtype=int)
    if selected.size == 0:
        return np.nan

    target_term = float(np.mean(avg_grm_to_target[selected]))
    block = train_train_grm[np.ix_(selected, selected)]
    if include_diagonal or selected.size == 1:
        internal_term = float(np.mean(block))
    else:
        internal_term = float((np.sum(block) - np.trace(block)) / (selected.size * (selected.size - 1)))
    return target_term - float(lambda_div) * internal_term


def subset_stats(selected, avg_grm_to_target, train_train_grm, lambda_div, include_diagonal=True):
    selected = np.asarray(selected, dtype=int)
    block = train_train_grm[np.ix_(selected, selected)]
    if include_diagonal or selected.size == 1:
        internal = float(np.mean(block))
    else:
        internal = float((np.sum(block) - np.trace(block)) / (selected.size * (selected.size - 1)))
    target = float(np.mean(avg_grm_to_target[selected]))
    return {
        "q": target - float(lambda_div) * internal,
        "avg_grm_to_target": target,
        "avg_internal_grm": internal,
    }


def greedy_avggrm_diversity(avg_grm_to_target, train_train_grm, k, lambda_div, include_diagonal=True):
    s = np.asarray(avg_grm_to_target, dtype=float)
    G = np.asarray(train_train_grm, dtype=float)
    n = s.size

    selected_mask = np.zeros(n, dtype=bool)
    relatedness_to_selected = np.zeros(n, dtype=float)
    diag = np.diag(G)
    target_sum = 0.0
    train_train_sum = 0.0
    order = []

    for step in range(int(k)):
        next_size = step + 1
        candidate_target_sum = target_sum + s
        candidate_avg_target = candidate_target_sum / next_size

        if include_diagonal:
            candidate_train_train_sum = train_train_sum + 2.0 * relatedness_to_selected + diag
            candidate_avg_train_train = candidate_train_train_sum / (next_size * next_size)
        else:
            candidate_train_train_sum = train_train_sum + 2.0 * relatedness_to_selected
            if next_size <= 1:
                candidate_avg_train_train = np.zeros(n, dtype=float)
            else:
                candidate_avg_train_train = candidate_train_train_sum / (next_size * (next_size - 1))

        candidate_q = candidate_avg_target - float(lambda_div) * candidate_avg_train_train
        candidate_q = np.where(selected_mask, -np.inf, candidate_q)
        best = int(np.argmax(candidate_q))

        order.append(best)
        selected_mask[best] = True
        target_sum = float(candidate_target_sum[best])
        train_train_sum = float(candidate_train_train_sum[best])
        relatedness_to_selected += G[:, best]

    return np.array(order, dtype=int)

## Genetic algorithm with simulated annealing

This section optimizes the same objective with the metaheuristic used by Fernandez-Gonzalez et al. (2023), built around TrainSel (Akdemir et al. 2021): a **genetic algorithm** for global search combined with **simulated annealing** for local refinement.

- each individual is a size-$k$ subset of the candidate pool,
- the population evolves through tournament selection, subset crossover, swap mutation, and elitism,
- every offspring is refined by a few simulated-annealing swap steps that accept a worsening swap with probability $\exp(\Delta Q / T)$,
- a best-improvement local search periodically polishes the incumbent and runs once more at the end,
- one individual is seeded with the greedy solution as a warm start, so GA+SA should not return a worse subset than greedy.

The objective is evaluated **incrementally**. Each subset carries a running vector $\mathrm{rel}[c] = \sum_{j \in S} G_{c,j}$; a single swap is then scored in $O(1)$ and committed in $O(n)$, instead of the $O(k^2)$ cost of recomputing $Q$ from scratch. Best-improvement local search becomes one vectorized $O(k\cdot n)$ pass over all candidate swaps. This is what makes the method practical for large $k$.


In [ ]:
def optimize_avggrm_diversity_ga_sa(
    avg_grm_to_target,
    train_train_grm,
    k,
    lambda_div,
    include_diagonal=True,
    pop_size=80,
    n_generations=250,
    sa_steps_per_gen=12,
    tournament_size=3,
    elite_frac=0.1,
    mutation_rate=0.6,
    mutation_swaps=2,
    polish_every=40,
    seed_solution=None,
    rng=None,
):
    """Genetic algorithm with simulated-annealing refinement for AvgGRM-diversity.

    Maximizes the same Q as ``q_value`` by evolving a population of size-``k``
    subsets, mirroring the TrainSel strategy of Fernandez-Gonzalez et al.
    (2023): a genetic algorithm explores globally while simulated-annealing
    swap steps refine every offspring and a best-improvement local search
    polishes the incumbent.

    The objective is evaluated incrementally. Each subset carries a state with
    a running vector ``rel[c] = sum_{j in S} G[c, j]``; a swap (drop one
    member, add one non-member) is then scored in O(1) and committed in O(n),
    instead of the O(k^2) cost of recomputing Q from scratch. This keeps the
    search practical at large k (e.g. n = 1000, k = 300).

    Returns the best subset (sorted local indices), its Q value, and the
    per-generation history of the best Q.
    """
    if rng is None:
        rng = np.random.default_rng()

    s = np.asarray(avg_grm_to_target, dtype=float).reshape(-1)
    G = np.asarray(train_train_grm, dtype=float)
    n = s.size
    k = int(k)
    if not (1 <= k <= n):
        raise ValueError("k must be between 1 and the number of candidates")
    diag_G = np.ascontiguousarray(np.diag(G), dtype=float)

    # Q = tsum / k - qcoef * quad, where quad excludes the diagonal when
    # include_diagonal is False (matching q_value).
    if include_diagonal:
        qcoef = lambda_div / (k * k)
        drop_diag = False
    else:
        qcoef = lambda_div / (k * (k - 1)) if k > 1 else 0.0
        drop_diag = True

    # --- incremental subset state -----------------------------------------
    # rel[c] = sum_{j in S} G[c, j]; qsum = sum_{i, j in S} G_ij (with diagonal).
    def make_state(subset):
        members = np.asarray(subset, dtype=int).copy()
        in_mask = np.zeros(n, dtype=bool)
        in_mask[members] = True
        rel = G[:, members].sum(axis=1)
        return {
            "members": members,
            "in_mask": in_mask,
            "tsum": float(s[members].sum()),
            "qsum": float(rel[members].sum()),
            "rel": rel,
        }

    def state_q(st):
        quad = st["qsum"]
        if drop_diag:
            quad = quad - float(diag_G[st["members"]].sum())
        return st["tsum"] / k - qcoef * quad

    def swap_delta(st, out_pos, in_elem):
        # Change in Q for dropping members[out_pos] and adding in_elem.
        out_elem = st["members"][out_pos]
        rel = st["rel"]
        d_quad = (-2.0 * rel[out_elem] + diag_G[out_elem]
                  + 2.0 * (rel[in_elem] - G[in_elem, out_elem]) + diag_G[in_elem])
        d_q = (s[in_elem] - s[out_elem]) / k - qcoef * d_quad
        if drop_diag:
            d_q += qcoef * (diag_G[in_elem] - diag_G[out_elem])
        return d_q

    def apply_swap(st, out_pos, in_elem):
        out_elem = st["members"][out_pos]
        rel = st["rel"]
        d_quad = (-2.0 * rel[out_elem] + diag_G[out_elem]
                  + 2.0 * (rel[in_elem] - G[in_elem, out_elem]) + diag_G[in_elem])
        st["qsum"] += d_quad
        st["tsum"] += s[in_elem] - s[out_elem]
        rel += G[:, in_elem] - G[:, out_elem]
        st["in_mask"][out_elem] = False
        st["in_mask"][in_elem] = True
        st["members"][out_pos] = in_elem

    def fitness(subset):
        return state_q(make_state(subset))

    def random_subset():
        return rng.choice(n, size=k, replace=False)

    def draw_non_member(in_mask):
        # Rejection sampling for a uniform non-member; k < n so this terminates fast.
        while True:
            cand = int(rng.integers(n))
            if not in_mask[cand]:
                return cand

    def anneal(subset, temp):
        st = make_state(subset)
        if k < n:
            for _ in range(sa_steps_per_gen):
                out_pos = int(rng.integers(k))
                in_elem = draw_non_member(st["in_mask"])
                delta = swap_delta(st, out_pos, in_elem)
                if delta >= 0.0 or rng.random() < np.exp(delta / temp):
                    apply_swap(st, out_pos, in_elem)
        return st["members"].copy(), state_q(st)

    def local_search(subset):
        # Best-improvement single-swap hill climbing, vectorized over all
        # (out_pos, in_elem) pairs in one O(k*n) pass.
        st = make_state(subset)
        while k < n:
            members = st["members"]
            rel = st["rel"]
            a = -2.0 * rel[members] + diag_G[members]
            b = 2.0 * rel + diag_G
            d_quad = a[:, None] + b[None, :] - 2.0 * G[members, :]
            d_q = (s[None, :] - s[members][:, None]) / k - qcoef * d_quad
            if drop_diag:
                d_q += qcoef * (diag_G[None, :] - diag_G[members][:, None])
            d_q[:, st["in_mask"]] = -np.inf
            flat = int(np.argmax(d_q))
            out_pos, in_elem = flat // n, flat % n
            if d_q[out_pos, in_elem] <= 1e-12:
                break
            apply_swap(st, out_pos, in_elem)
        return st["members"].copy(), state_q(st)

    # --- annealing temperature schedule -----------------------------------
    # Calibrate to the scale of single-swap |delta Q| seen from random subsets,
    # so early SA steps can genuinely escape local optima.
    deltas = []
    if k < n:
        for _ in range(25):
            probe = make_state(random_subset())
            for _ in range(12):
                out_pos = int(rng.integers(k))
                in_elem = draw_non_member(probe["in_mask"])
                deltas.append(abs(swap_delta(probe, out_pos, in_elem)))
    positive = [d for d in deltas if d > 0]
    scale = float(np.mean(positive)) if positive else 1e-6
    temps = np.geomspace(scale, scale * 1e-3, num=max(n_generations, 1))

    # --- initial population ------------------------------------------------
    population = [random_subset() for _ in range(pop_size)]
    if seed_solution is not None:
        population[0] = np.asarray(seed_solution, dtype=int).copy()
    fitness_values = np.array([fitness(ind) for ind in population], dtype=float)

    # --- genetic operators -------------------------------------------------
    def tournament():
        contenders = rng.choice(pop_size, size=min(tournament_size, pop_size), replace=False)
        return population[contenders[int(np.argmax(fitness_values[contenders]))]]

    def crossover(parent_a, parent_b):
        set_a, set_b = set(parent_a.tolist()), set(parent_b.tolist())
        common = np.array(sorted(set_a & set_b), dtype=int)
        pool = np.array(sorted(set_a ^ set_b), dtype=int)
        rng.shuffle(pool)
        return np.concatenate([common, pool[: k - common.size]])

    def mutate(individual):
        if k == n:
            return individual
        child = individual.copy()
        in_mask = np.zeros(n, dtype=bool)
        in_mask[child] = True
        not_in = np.flatnonzero(~in_mask)
        n_swap = min(mutation_swaps, k, not_in.size)
        out_pos = rng.choice(k, size=n_swap, replace=False)
        child[out_pos] = rng.choice(not_in, size=n_swap, replace=False)
        return child

    # --- evolution loop ----------------------------------------------------
    best = population[int(np.argmax(fitness_values))].copy()
    best_fit = float(np.max(fitness_values))
    history = [best_fit]
    elite_count = max(1, int(round(elite_frac * pop_size)))

    for gen in range(n_generations):
        temp = float(temps[gen])
        order = np.argsort(fitness_values)[::-1]
        new_population = [population[i].copy() for i in order[:elite_count]]
        new_fitness = [float(fitness_values[i]) for i in order[:elite_count]]
        while len(new_population) < pop_size:
            child = crossover(tournament(), tournament())
            if rng.random() < mutation_rate:
                child = mutate(child)
            child, child_fit = anneal(child, temp)
            new_population.append(child)
            new_fitness.append(child_fit)
        population = new_population
        fitness_values = np.array(new_fitness, dtype=float)

        gen_best = int(np.argmax(fitness_values))
        if fitness_values[gen_best] > best_fit:
            best_fit = float(fitness_values[gen_best])
            best = population[gen_best].copy()

        # Memetic polish: drive the incumbent to a local optimum and feed it
        # back into the population so crossover can spread its good elements.
        if polish_every and (gen + 1) % polish_every == 0:
            polished, polished_fit = local_search(best)
            if polished_fit > best_fit:
                best_fit, best = polished_fit, polished.copy()
            worst = int(np.argmin(fitness_values))
            population[worst] = polished
            fitness_values[worst] = polished_fit

        history.append(best_fit)

    best, best_fit = local_search(best)
    history.append(best_fit)
    return np.sort(best), best_fit, np.array(history, dtype=float)

In [ ]:
# GA+SA hyperparameters. Roughly mirrors the TrainSel settings in
# Fernandez-Gonzalez et al. (2023): a few hundred GA generations, each
# refined by a handful of simulated-annealing swap steps. Tuned upward from
# the small-experiment defaults (250 gens / 2-swap mutation / polish every 40)
# because the n = 1000, k = 300 run showed repeat-to-repeat variance in the
# gain at high lambda. GA_MUTATION_SWAPS is scaled with k so that mutation
# moves stay around 1.5-2% of the subset size (5 at k=300, 10 at k=600).
GA_POP_SIZE = 80
GA_GENERATIONS = 400
GA_SA_STEPS = 12
GA_TOURNAMENT = 3
GA_ELITE_FRAC = 0.1
GA_MUTATION_RATE = 0.7
GA_MUTATION_SWAPS = 10
GA_POLISH_EVERY = 25
GA_SEED_WITH_GREEDY = True

sub_rng = np.random.default_rng(SEED)
ga_rows = []
ga_selected_sets = []
ga_run_start = time.perf_counter()

total_ga_jobs = N_REPEATS * len(LAMBDA_DIVS) * len(K_VALUES)
print(
    f"Running {total_ga_jobs} GA+SA optimizations: "
    f"N_CANDIDATES={N_CANDIDATES}, k={K_VALUES}, "
    f"pop={GA_POP_SIZE}, generations={GA_GENERATIONS}, sa_steps={GA_SA_STEPS}, "
    f"mutation_rate={GA_MUTATION_RATE}, mutation_swaps={GA_MUTATION_SWAPS}, "
    f"polish_every={GA_POLISH_EVERY}, seed_with_greedy={GA_SEED_WITH_GREEDY}",
    flush=True,
)

for repeat in range(N_REPEATS):
    sampled_global_idx = sub_rng.choice(source_idx_all, size=N_CANDIDATES, replace=False)
    sample_ids = ids[sampled_global_idx]

    avg_grm_to_target = grm[np.ix_(sampled_global_idx, target_idx)].mean(axis=1)
    train_train_grm = grm[np.ix_(sampled_global_idx, sampled_global_idx)]

    for lambda_div in LAMBDA_DIVS:
        for k in K_VALUES:
            g_t0 = time.perf_counter()
            greedy_sel = greedy_avggrm_diversity(
                avg_grm_to_target,
                train_train_grm,
                k=k,
                lambda_div=lambda_div,
                include_diagonal=INCLUDE_DIAGONAL,
            )
            greedy_time = time.perf_counter() - g_t0

            # Deterministic per-setting RNG so the section is reproducible.
            ga_rng = np.random.default_rng([SEED, repeat, int(round(lambda_div * 1000)), int(k)])
            start = time.perf_counter()
            ga_sel, ga_fit, ga_history = optimize_avggrm_diversity_ga_sa(
                avg_grm_to_target,
                train_train_grm,
                k=k,
                lambda_div=lambda_div,
                include_diagonal=INCLUDE_DIAGONAL,
                pop_size=GA_POP_SIZE,
                n_generations=GA_GENERATIONS,
                sa_steps_per_gen=GA_SA_STEPS,
                tournament_size=GA_TOURNAMENT,
                elite_frac=GA_ELITE_FRAC,
                mutation_rate=GA_MUTATION_RATE,
                mutation_swaps=GA_MUTATION_SWAPS,
                polish_every=GA_POLISH_EVERY,
                seed_solution=greedy_sel if GA_SEED_WITH_GREEDY else None,
                rng=ga_rng,
            )
            ga_time = time.perf_counter() - start

            greedy_stats = subset_stats(greedy_sel, avg_grm_to_target, train_train_grm, lambda_div, INCLUDE_DIAGONAL)
            ga_stats = subset_stats(ga_sel, avg_grm_to_target, train_train_grm, lambda_div, INCLUDE_DIAGONAL)

            ga_set = set(map(int, ga_sel))
            greedy_set = set(map(int, greedy_sel))
            overlap_greedy = len(ga_set & greedy_set)
            union_greedy = len(ga_set | greedy_set)

            target_gain = ga_stats["avg_grm_to_target"] - greedy_stats["avg_grm_to_target"]
            internal_reduction = greedy_stats["avg_internal_grm"] - ga_stats["avg_internal_grm"]
            q_relative_gain = (ga_stats["q"] - greedy_stats["q"]) / max(abs(greedy_stats["q"]), 1e-12)

            row = {
                "repeat": repeat,
                "target_code": target_code,
                "target_label": code_to_label[target_code],
                "n_candidates": N_CANDIDATES,
                "k": int(k),
                "lambda_div": float(lambda_div),
                "q_greedy": greedy_stats["q"],
                "q_ga": ga_stats["q"],
                "q_ga_minus_greedy": ga_stats["q"] - greedy_stats["q"],
                "q_relative_gain": q_relative_gain,
                "avg_target_greedy": greedy_stats["avg_grm_to_target"],
                "avg_target_ga": ga_stats["avg_grm_to_target"],
                "target_gain_ga_minus_greedy": target_gain,
                "avg_internal_greedy": greedy_stats["avg_internal_grm"],
                "avg_internal_ga": ga_stats["avg_internal_grm"],
                "internal_reduction_ga_vs_greedy": internal_reduction,
                "ga_overlap_greedy": overlap_greedy,
                "ga_overlap_frac_greedy": overlap_greedy / float(k),
                "ga_jaccard_greedy": overlap_greedy / float(union_greedy) if union_greedy else np.nan,
                "greedy_time_sec": greedy_time,
                "ga_time_sec": ga_time,
                "ga_time_per_generation_ms": 1000.0 * ga_time / max(GA_GENERATIONS, 1),
                "ga_generations": GA_GENERATIONS,
            }

            ga_rows.append(row)
            ga_selected_sets.append({
                "repeat": repeat,
                "k": int(k),
                "lambda_div": float(lambda_div),
                "greedy_local_idx": greedy_sel,
                "ga_local_idx": ga_sel,
                "greedy_ids": sample_ids[greedy_sel],
                "ga_ids": sample_ids[ga_sel],
                "ga_history": ga_history,
            })

            elapsed = time.perf_counter() - ga_run_start
            print(
                f"repeat={repeat}, lambda={lambda_div:g}, k={k}: "
                f"q_ga={ga_stats['q']:.6f}, q_greedy={greedy_stats['q']:.6f}, "
                f"gain={ga_stats['q'] - greedy_stats['q']:+.3e} ({q_relative_gain*100:+.2f}%), "
                f"jaccard_to_greedy={row['ga_jaccard_greedy']:.3f}, "
                f"greedy_time={greedy_time*1000:.1f} ms, "
                f"ga_time={ga_time:.2f}s, elapsed={elapsed:.1f}s",
                flush=True,
            )

ga_results_df = pd.DataFrame(ga_rows)
ga_results_path = RESULTS_DIR / f"{PLOT_STEM}_results.csv"
ga_results_df.to_csv(ga_results_path, index=False)

selected_payload = []
for item in ga_selected_sets:
    selected_payload.append({
        "repeat": int(item["repeat"]),
        "k": int(item["k"]),
        "lambda_div": float(item["lambda_div"]),
        "greedy_local_idx": np.asarray(item["greedy_local_idx"], dtype=int).tolist(),
        "ga_local_idx": np.asarray(item["ga_local_idx"], dtype=int).tolist(),
        "greedy_ids": np.asarray(item["greedy_ids"], dtype=str).tolist(),
        "ga_ids": np.asarray(item["ga_ids"], dtype=str).tolist(),
        "ga_history": np.asarray(item["ga_history"], dtype=float).tolist(),
    })
selected_sets_path = RESULTS_DIR / f"{PLOT_STEM}_selected_sets.json"
selected_sets_path.write_text(json.dumps(selected_payload), encoding="utf-8")

display(ga_results_df.head())
print(f"Finished {len(ga_results_df)} GA+SA runs in {time.perf_counter() - ga_run_start:.1f}s")
print(f"Saved results to {ga_results_path}")
print(f"Saved selected sets to {selected_sets_path}")


In [ ]:
if "ga_results_df" not in globals():
    stored_results_path = RESULTS_DIR / f"{PLOT_STEM}_results.csv"
    ga_results_df = pd.read_csv(stored_results_path)
    print(f"Loaded stored results from {stored_results_path}")

agg = {
    "n": ("repeat", "count"),
    "q_greedy_mean": ("q_greedy", "mean"),
    "q_ga_mean": ("q_ga", "mean"),
    "ga_gain_mean": ("q_ga_minus_greedy", "mean"),
    "ga_gain_sd": ("q_ga_minus_greedy", "std"),
    "ga_gain_max": ("q_ga_minus_greedy", "max"),
    "ga_relative_gain_pct_mean": ("q_relative_gain", lambda x: 100.0 * x.mean()),
    "ga_jaccard_greedy_mean": ("ga_jaccard_greedy", "mean"),
    "greedy_time_ms_mean": ("greedy_time_sec", lambda x: 1000.0 * x.mean()),
    "ga_time_sec_mean": ("ga_time_sec", "mean"),
}
optional_agg = {
    "avg_target_greedy_mean": ("avg_target_greedy", "mean"),
    "avg_target_ga_mean": ("avg_target_ga", "mean"),
    "target_gain_mean": ("target_gain_ga_minus_greedy", "mean"),
    "avg_internal_greedy_mean": ("avg_internal_greedy", "mean"),
    "avg_internal_ga_mean": ("avg_internal_ga", "mean"),
    "internal_reduction_mean": ("internal_reduction_ga_vs_greedy", "mean"),
    "ga_overlap_frac_greedy_mean": ("ga_overlap_frac_greedy", "mean"),
}
for out_col, spec in optional_agg.items():
    if spec[0] in ga_results_df.columns:
        agg[out_col] = spec

ga_summary = ga_results_df.groupby(["lambda_div", "k"], as_index=False).agg(**agg)

ga_summary["ga_slowdown_vs_greedy"] = (
    ga_summary["ga_time_sec_mean"] / (ga_summary["greedy_time_ms_mean"] / 1000.0)
)

ga_summary_path = RESULTS_DIR / f"{PLOT_STEM}_summary.csv"
ga_summary.to_csv(ga_summary_path, index=False)

display(ga_summary)
print(f"Saved summary to {ga_summary_path}")


In [ ]:
if "ga_results_df" not in globals():
    stored_results_path = RESULTS_DIR / f"{PLOT_STEM}_results.csv"
    ga_results_df = pd.read_csv(stored_results_path)
    print(f"Loaded stored results from {stored_results_path}")

plot_module_path = PROJECT_ROOT / "scripts" / "plot_avggrm_diversity_ga_sa.py"
spec = importlib.util.spec_from_file_location("plot_avggrm_diversity_ga_sa", plot_module_path)
plot_avggrm_diversity_ga_sa = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = plot_avggrm_diversity_ga_sa
spec.loader.exec_module(plot_avggrm_diversity_ga_sa)

local_plot_paths = plot_avggrm_diversity_ga_sa.plot_avggrm_diversity_ga_comparison(
    ga_results_df,
    output_dir=FIGURES_DIR,
    stem=PLOT_STEM,
    repo_root=PROJECT_ROOT,
)

thesis_plot_paths = None
if THESIS_FIGURES_DIR.exists():
    thesis_plot_paths = plot_avggrm_diversity_ga_sa.plot_avggrm_diversity_ga_comparison(
        ga_results_df,
        output_dir=THESIS_FIGURES_DIR,
        stem=PLOT_STEM,
        repo_root=PROJECT_ROOT,
    )

{
    "local_plot_paths": local_plot_paths,
    "thesis_plot_paths": thesis_plot_paths,
}
